# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`

This notebook demonstrates how to load, inspect, and perform exploratory analysis on the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. This dataset contains clinicopathological and molecular characteristics of cancer survivors diagnosed with second primary colorectal cancer, including key variables such as anatomical distribution, biomarker status, comorbidities, and more.

### Dataset Source
The dataset is defined using a [Croissant schema](https://mlcommons.org/croissant/) accessible at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

We'll use `mlcroissant` to load the dataset metadata. This allows us to inspect the dataset structure before working with the actual data records.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset using the Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access and print brief summary (name and description)
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview

Let's inspect the record sets, fields, and columns defined in this dataset. All entities are referenced by their Croissant `@id`, as recommended for robust and schema-consistent access.

We'll print the available record sets and the fields within each.

In [ ]:
# Print available record sets and their field @ids
print("Available record sets:")
record_sets = list(dataset.list_record_sets())
for recset in record_sets:
    print(f"  - RecordSet @id: {recset['@id']}, name: {recset.get('name', 'N/A')}")
    print("    Fields:")
    for field in recset.get('field', []):
        print(f"      - Field @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")


## 3. Data Extraction

Now let's extract the records from each record set using their `@id` values. We'll load each record set into a separate DataFrame, using the Croissant `@id` for referencing. This enables easy and explicit downstream processing.

In [ ]:
# List all record set @ids
record_sets = [recset['@id'] for recset in dataset.list_record_sets()]
dataframes = {}

for recset_id in record_sets:
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)

# For demonstration, pick the first record set
if record_sets:
    preview_recset_id = record_sets[0]
    print(f"Columns in record set '{preview_recset_id}':")
    print(dataframes[preview_recset_id].columns.tolist())
    display(dataframes[preview_recset_id].head())

### Record Set and Field Details

The first record set loaded above (`preview_recset_id`) can be explored further below. You may want to adjust the following analysis to focus on fields relevant to your analysis or of numeric type for EDA.

## 4. Exploratory Data Analysis (EDA)

Let's perform data processing steps on a numeric field (e.g., age or interval fields). This section demonstrates filtering records, normalization, outlier handling, and group summarization. All columns are referenced by their `@id`.

We'll:
1. Select a numeric field by inspecting the available columns.
2. Filter records above a threshold value.
3. Normalize the selected numeric field.
4. Optionally group by a relevant categorical variable.

In [ ]:
# Inspect columns to select a numeric field (adjust as suitable for your data)
df = dataframes[preview_recset_id]
print("Available columns in this record set:")
for c in df.columns:
    print(c)

# Example: Try to select a numeric field by @id, such as an age or diagnosis interval column
# Please update these according to the real column @ids shown above

# Attempt to pick common medical field @ids; replace with your dataset's values
candidate_columns = [c for c in df.columns if "interval" in c.lower() or "age" in c.lower() or "number" in c.lower()]
if candidate_columns:
    numeric_field_id = candidate_columns[0]
else:
    # Fallback to the first column
    numeric_field_id = df.columns[0]

print(f"\nSelected numeric field for analysis: {numeric_field_id}")
# Ensure this is numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Threshold for filtering (choose a value that makes sense, e.g., 10 years interval)
threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().sum() > 0 else 10

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() is not None and filtered_df[numeric_field_id].std() != 0 else 0
)

print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by the next available categorical field
categorical_candidates = [c for c in df.columns if c != numeric_field_id and (df[c].dtype=='object')]
group_field_id = categorical_candidates[0] if categorical_candidates else None

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relationship with a categorical variable if available (e.g., using boxplots or histograms).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, color='skyblue')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot grouped by a categorical field if available
if group_field_id:
    plt.figure(figsize=(10,5))
    # Ensure the group_field_id is string type
    sns.boxplot(x=df[group_field_id].astype(str), y=df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:

- Load FAIR^2 dataset metadata and records using Croissant schemas and the `mlcroissant` library.
- Explore dataset structure, identifying record sets and referencing all entities by their stable Croissant `@id`s.
- Extract data into DataFrames, filter and normalize a numeric field, and compute groupwise statistics.
- Visualize key features, empowering further clinical or biomarker analysis on cancer survivor cohorts.

This approach ensures reproducible, schema-driven data science workflows for modern FAIR datasets. Please adapt field IDs and record set selections as needed for deeper analyses.
